In [39]:
import os
import json
import numpy as np
import pandas as pd

from IPython.core.completerlib import try_import
from pandas.conftest import skipna

In [22]:
base = ".."+os.sep+"data"
file = "jobs_data_export.json"
with open(f"{base}{os.sep}{file}", 'r') as f:
    data = json.load(f)

jobs = pd.read_csv(f"{base}{os.sep}users_data_export.csv")

In [23]:
print(jobs.head())
#data

                        _id         now  user_id  arrival  processing  expire  \
0  6901e0290e7dfe76e0a2ec4d    0.000000        0        1          17       6   
1  6901e0290e7dfe76e0a2f72f   77.417021        1       78          96       5   
2  6901e02a0e7dfe76e0a30727  189.706629        2      190         206       9   
3  6901e02a0e7dfe76e0a30f7a  248.701610        3      249         263       8   
4  6901e02a0e7dfe76e0a30f7b  248.722826        4      249         269       9   

       cpu     mem     hdd         x          y  
0   1500.0   200.0   32.00  2.208430  41.410557  
1   1500.0   200.0   32.00  2.200862  41.413023  
2   7500.0   200.0    1.28  2.197146  41.409687  
3   3000.0   800.0    1.28  2.203465  41.411232  
4  30000.0  4000.0  128.00  2.204719  41.408507  


In [24]:
# por cada job extraer el tiempo que llega, las caracteristicas, el primer serividor asignado y el tiempo que tarda en completar o expirar
jobs_dict = {}
for record in data:
    event = record['status']
    timestamp = record['time']
    if event in ['TypeEvent.user_arrival','TypeEvent.user_server_assigned','TypeEvent.user_completed','TypeEvent.user_expired']:
        job_id = int(record['origin_id'].split()[1])
        if event == 'TypeEvent.user_arrival':
            if job_id not in jobs_dict:
                jobs_dict[job_id] = {
                'time_arrival': timestamp,
                'first_server_assigned': None,
                'time_finishing': None}|jobs[jobs['user_id']==job_id][['arrival','processing','expire','cpu','mem','hdd','x','y']].iloc[0].to_dict()
        elif event == 'TypeEvent.user_server_assigned' and jobs_dict[job_id]['first_server_assigned'] is None:
            jobs_dict[job_id]['first_server_assigned'] = record['destination_id'].split()[1]
        elif (event == 'TypeEvent.user_completed' or event == 'TypeEvent.user_expired'):
            jobs_dict[job_id]['time_finishing'] = timestamp

df_jobs_dict = pd.DataFrame.from_dict(jobs_dict,orient='index')

'''
    if event == 'user_arrival':
        jobs_dict[job_id]['arrival_time'] = timestamp
        jobs_dict[job_id]['features'] = record.get('features', {})

    elif event == 'user_server_assigned' and jobs_dict[job_id]['first_server_assigned'] is None:
        jobs_dict[job_id]['first_server_assigned'] = record.get('server_id')

    elif event == 'user_completed':
        jobs_dict[job_id]['completion_time'] = timestamp

    elif event == 'user_expired':
        jobs_dict[job_id]['expiration_time'] = timestamp'''

# create dataframe where key index and each vales, that is dictiornary use keys as columns and values as rows


"\n    if event == 'user_arrival':\n        jobs_dict[job_id]['arrival_time'] = timestamp\n        jobs_dict[job_id]['features'] = record.get('features', {})\n\n    elif event == 'user_server_assigned' and jobs_dict[job_id]['first_server_assigned'] is None:\n        jobs_dict[job_id]['first_server_assigned'] = record.get('server_id')\n\n    elif event == 'user_completed':\n        jobs_dict[job_id]['completion_time'] = timestamp\n\n    elif event == 'user_expired':\n        jobs_dict[job_id]['expiration_time'] = timestamp"

In [25]:
df_jobs_dict.to_csv(f"{base}{os.sep}summary.csv", index=False)

In [47]:
# convert types to in
df_jobs_dict["first_server_assigned"].fillna(np.nan, inplace=True)

In [58]:
#df_jobs_dict.isna().sum()
df_jobs_dict["first_server_assigned"].fillna(df_jobs_dict["first_server_assigned"].mode()[0],inplace=True)

In [61]:
df_jobs_dict[df_jobs_dict['time_finishing'].isna()]['time_finishing']

77833    None
77834    None
Name: time_finishing, dtype: object

In [77]:
# Correct iloc usage: select column then position
df_jobs_dict['time_arrival'].iloc[77833] + df_jobs_dict['expire'].iloc[77833]

SyntaxError: incomplete input (1646744182.py, line 1)